In [1]:
%load_ext autoreload
%autoreload 2

import os

import requests
import pandas as pd
from dotenv import load_dotenv
from gasp import db

load_dotenv()
API_KEY = os.getenv("EIA_API_KEY")

In [2]:
url = "https://api.eia.gov/v2/natural-gas/stor/wkly/data/"     # TODO: everything before the ? in your browser url

params = {
    "api_key": API_KEY,
    "frequency": "weekly",
    "data[0]": "value",
    "facets[series][]": "NW2_EPG0_SWO_R48_BCF",
    "sort[0][column]": "period",
    "sort[0][direction]": "asc",
    "offset": 0,
    "length": 5000
}

r = requests.get(url, params=params)     # TODO: send the GET request with url and params
r.raise_for_status()                     # TODO: raise an error if the request failed

response = r.json()                      # TODO: turn the response into a python dict
rows = response["response"]["data"]                  # TODO: the list of rows inside response

len(rows)

876

In [3]:
print("Number of rows:", len(rows))
print("First row:", rows[0] if rows else "No data returned")

Number of rows: 876
First row: {'period': '2010-01-01', 'duoarea': 'R48', 'area-name': 'NA', 'product': 'EPG0', 'product-name': 'Natural Gas', 'process': 'SWO', 'process-name': 'Underground Storage - Working Gas', 'series': 'NW2_EPG0_SWO_R48_BCF', 'series-description': 'Weekly Lower 48 States Natural Gas Working Underground Storage (Billion Cubic Feet)', 'value': '3117', 'units': 'BCF'}


In [4]:
df = pd.DataFrame(rows)[["period", "value"]]
df["period"] = pd.to_datetime(df["period"])
df["value"] = pd.to_numeric(df["value"]).astype(float)
df["series"] = "NW2_EPG0_SWO_R48_BCF"
df = df[["period", "series", "value"]].sort_values("period").reset_index(drop=True)

df.head()

,period,series,value
0,2010-01-01,NW2_EPG0_SWO_R48_BCF,3117.0
1,2010-01-08,NW2_EPG0_SWO_R48_BCF,2850.0
2,2010-01-15,NW2_EPG0_SWO_R48_BCF,2607.0
3,2010-01-22,NW2_EPG0_SWO_R48_BCF,2521.0
4,2010-01-29,NW2_EPG0_SWO_R48_BCF,2406.0


In [5]:
db.init_db()

first_write = db.write_series(df)
second_write = db.write_series(df)

print("First write:", first_write)
print("Second write:", second_write)

First write: 876
Second write: 0


In [6]:
# Create a copy of the original data
revision_df = df.iloc[[0]].copy()

# Change one value to simulate an EIA correction
revision_df.loc[revision_df.index[0], "value"] += 100.0

# Use a separate test series so existing data isn't affected
revision_df["series"] = "REVISION_DEMO"

original_df = df.iloc[[0]].copy()
original_df["series"] = "REVISION_DEMO"

print("Original value:", original_df["value"].iloc[0])
print("Revised value:", revision_df["value"].iloc[0])

Original value: 3117.0
Revised value: 3217.0


In [7]:
# Save original measurement as retrieved on January 10
original_count = db.write_series(
    original_df,
    retrieved_at="2024-01-10"
)

# Save revised measurement as retrieved on January 20
revision_count = db.write_series(
    revision_df,
    retrieved_at="2024-01-20"
)

print("Original rows added:", original_count)
print("Revision rows added:", revision_count)

Original rows added: 1
Revision rows added: 1


In [8]:
before = db.read_series(
    "REVISION_DEMO",
    as_of="2024-01-15"
)

after = db.read_series(
    "REVISION_DEMO",
    as_of="2024-01-25"
)

print("Before revision:")
display(before)

print("After revision:")
display(after)

Before revision:


,period,series,value
0,2010-01-01,REVISION_DEMO,3117.0


After revision:


,period,series,value
0,2010-01-01,REVISION_DEMO,3217.0
